# N1 - Clean and normalize all record files

Kaggle settings: **Internet ON**, accelerator **CPU (no accelerator)**. Add inputs: `amz-er-2026-raw` + `amz-er-2026-code` + previous stage outputs. See `kaggle/PUSH_INSTRUCTIONS.md` in the repo.

In [ ]:
!pip -q install anyascii jellyfish rapidfuzz pyarrow

In [ ]:
import glob
import os
import subprocess
import sys

CODE = "/kaggle/input/amz-er-2026-code/src"
RAW = "/kaggle/input/amz-er-2026-raw"
ART = "/kaggle/working/artifacts"
os.environ["BER_DATA_DIR"] = RAW
os.environ["BER_ARTIFACT_DIR"] = ART
sys.path.insert(0, CODE)


def find(sub):
    hits = sorted(glob.glob(f"/kaggle/input/**/artifacts/{sub}", recursive=True))
    print(sub, "->", hits[:3])
    return hits[0] if hits else ""


CLEAN = find("clean")
BLOCK = find("block")
EMBED = find("embed")
GBDT = find("gbdt")
RERANK = find("rerank")
print("inputs located")


def run(module, *args):
    subprocess.run([sys.executable, "-m", module, *args], check=True)

In [ ]:
run("ber.stages.clean")

In [ ]:
import json
from pathlib import Path

for sub in ("clean", "block", "embed", "gbdt", "rerank", "out"):
    for name in ("metrics.json", "stats.json"):
        p = Path(ART) / sub / name
        if p.exists():
            print("==", sub, name, "==")
            print(p.read_text(encoding="utf-8"))

out_dir = Path(ART) / "out"
if out_dir.exists():
    print("outputs:", sorted(x.name for x in out_dir.glob("*.tsv")))